# 2. Model Training

**Purpose**: Train Splink model with full visibility. Checkpoints model for reuse.

## Sections
1. Load config and utils
2. Load training data
3. Create ground truth pairs (ROR matching)
4. Define Splink settings (comparisons, blocking)
5. Initialize Linker
6. Estimate u-probabilities
7. EM training
8. Validation on held-out pairs
9. Save model
10. Export training report


In [ ]:
pip install jellyfish

In [ ]:
pip install cleanco countryinfo

In [ ]:
 pip install pycountry geonamescache

---
## 1. Setup and Configuration


In [1]:
import sys
sys.path.insert(0, '/Users/robertlalani/Desktop/entity_resolution_12_18_25/org_claude_test')
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import pycountry
# Splink imports
from splink import Linker, SettingsCreator, block_on, DuckDBAPI
import splink.comparison_library as cl
import splink.comparison_level_library as cll

# Local imports
from config import config
from utils import (
    DatabaseManager, 
    log_step, 
    Timer,
    ProgressTracker,
    save_checkpoint,
    load_checkpoint,
    save_json,
    describe_dataframe
)
from data_prep import (
    load_and_prepare_training_data,
    create_ground_truth_pairs,
    create_blocking_keys,
    add_idf_based_features,
    add_token_set_features,
    compute_token_statistics,
    get_corpus_stopwords
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

print("Imports loaded successfully")


Imports loaded successfully


In [2]:
# Display configuration
print("TRAINING CONFIGURATION")
print("=" * 50)
print(f"dim_org sample size:    {config.sampling.TRAINING_DIM_ORG_SAMPLE:,}")
print(f"GRID sample size:       {config.sampling.TRAINING_GRID_SAMPLE:,}")
print(f"Validation holdout:     {config.sampling.VALIDATION_HOLDOUT_RATIO*100:.0f}%")
print(f"Prediction threshold:   {config.matching.THRESHOLD_PREDICTION}")
print(f"Model output path:      {config.paths.MODEL_FILE}")


TRAINING CONFIGURATION
dim_org sample size:    110,000
GRID sample size:       110,000
Validation holdout:     20%
Prediction threshold:   0.5
Model output path:      /Users/robertlalani/Desktop/entity_resolution_12_18_25/org_claude_test/models/model_v1.json


In [3]:
# Initialize database
db = DatabaseManager()
print("Database manager ready")


Database manager ready


---
## 2. Load Training Data


In [4]:
# Check for cached training data
cached_dim_org = load_checkpoint(config.paths.DATA_DIR + "/dim_org_training.parquet", "dim_org training cache")
cached_grid = load_checkpoint(config.paths.DATA_DIR + "/grid_training.parquet", "GRID training cache")

if cached_dim_org is not None and cached_grid is not None:
    print("Using cached training data")
    dim_org_df = create_blocking_keys(cached_dim_org)
    grid_df = create_blocking_keys(cached_grid)
else:
    print("Loading fresh training data from database...")
    dim_org_df, grid_df = load_and_prepare_training_data(
        db,
        dim_org_sample=config.sampling.TRAINING_DIM_ORG_SAMPLE,
        grid_sample=config.sampling.TRAINING_GRID_SAMPLE
    )
    
    # Cache for future runs
    save_checkpoint(dim_org_df, config.paths.DATA_DIR + "/dim_org_training.parquet", "dim_org training")
    save_checkpoint(grid_df, config.paths.DATA_DIR + "/grid_training.parquet", "GRID training")


[13:56:01]  Loading checkpoint: dim_org training cache
[13:56:01]    Loaded 109,851 rows
[13:56:01] [!] Checkpoint not found: /Users/robertlalani/Desktop/entity_resolution_12_18_25/org_claude_test/data/grid_training.parquet
Loading fresh training data from database...
[13:56:01]  Starting: Loading and preparing training data
[13:56:01]  Starting: Loading dim_organization
[13:56:01]  Executing: dim_organization (limit=110000)
[13:56:10]    Returned 110,000 rows in 9.2s
[13:56:10]  Completed: Loading dim_organization (9.4s)
[13:56:10]  Starting: Loading GRID data
[13:56:10]  Executing: GRID (limit=110000)
[13:56:18]    Returned 110,000 rows in 7.6s
[13:56:18]  Completed: Loading GRID data (7.6s)
[13:56:18]  Creating unified schema for dim_org...
[13:56:24]    Created unified schema: 110,000 rows, 16 columns
[13:56:24]  Creating unified schema for grid...
[13:56:29]    Created unified schema: 110,000 rows, 16 columns
[13:56:29]  Filtering bad records...
[13:56:29]    Removed 149 of 110,00

In [5]:
# Display training data summary
print("\nTRAINING DATA SUMMARY")
print("=" * 50)
describe_dataframe(dim_org_df, "dim_organization")
describe_dataframe(grid_df, "GRID")



TRAINING DATA SUMMARY

dim_organization
Shape: 109,851 rows x 20 columns

Column types:
  unique_id                      object            0.0% null
  name                           object            0.0% null
  name_clean                     object            0.0% null
  name_normalized                object            0.0% null
  name_prefix_5                  object            0.0% null
  name_prefix_10                 object            0.0% null
  all_names                      object            0.0% null
  org_type                       object            1.4% null
  country_code                   object            0.9% null
  city                           object            1.4% null
  latitude                       float64           2.0% null
  longitude                      float64           2.0% null
  ror_id                         object            3.9% null
  grid_id                        object            6.3% null
  source                         object            0.0% n

In [6]:
# Add IDF-based distinctive token features
# This automatically identifies corpus stopwords and extracts distinctive tokens
print("ADDING IDF-BASED FEATURES")
print("=" * 50)

[dim_org_df, grid_df], idf_scores, corpus_stopwords = add_idf_based_features(
    [dim_org_df, grid_df],
    stopword_percentile=0.25  # Bottom 25% IDF = more aggressive stopword filtering
)

print(f"\nCorpus statistics:")
print(f"  Total unique tokens:   {len(idf_scores):,}")
print(f"  Auto-identified stopwords: {len(corpus_stopwords)}")
print(f"\nSample stopwords (most common in corpus):")
sorted_stopwords = sorted(corpus_stopwords, key=lambda t: idf_scores.get(t, 0))[:30]
print(f"  {', '.join(sorted_stopwords)}")

# Show example distinctive tokens
print(f"\nExample distinctive tokens extraction:")
for i, row in dim_org_df.sample(5, random_state=42).iterrows():
    name = row['name_normalized'][:50] if row['name_normalized'] else 'N/A'
    tokens = row.get('distinctive_tokens', [])
    print(f"  '{name}...' -> {tokens}")

# Add token set features for containment detection
print(f"\nADDING TOKEN SET FEATURES")
print("=" * 50)
dim_org_df = add_token_set_features(dim_org_df, stopwords=corpus_stopwords)
grid_df = add_token_set_features(grid_df, stopwords=corpus_stopwords)

# Show example token sets
print(f"\nExample token sets for containment detection:")
for i, row in dim_org_df.sample(3, random_state=42).iterrows():
    name = row['name_normalized'][:40] if row['name_normalized'] else 'N/A'
    tokens = row.get('name_tokens', [])
    print(f"  '{name}...' -> {tokens[:5]}{'...' if len(tokens) > 5 else ''}")


ADDING IDF-BASED FEATURES
[13:56:30]  Adding IDF-based features to 2 DataFrames...
[13:56:30]  Computing token IDF statistics...
[13:56:30]    Computed IDF for 70,622 unique tokens
[13:56:30]    IDF range: 1.91 (most common) to 12.30 (most rare)
[13:56:30]    Auto-identified 19930 corpus stopwords (IDF <= 11.20)
[13:56:30]    Top 20 corpus stopwords: of, university, institute, and, hospital, for, research, de, center, foundation, college, national, medical, health, technology, the, centre, association, society, science
[13:56:30]  Adding distinctive tokens...
[13:56:31]    Loaded 34,655 geographic stopwords
[13:56:31]    Geographic stopwords: 34,655 (locations filtered out)
[13:56:31]    Distinctive token coverage: 99.9%
[13:56:31]  Adding distinctive tokens...
[13:56:31]    Geographic stopwords: 34,655 (locations filtered out)
[13:56:32]    Distinctive token coverage: 99.9%

Corpus statistics:
  Total unique tokens:   70,622
  Auto-identified stopwords: 19930

Sample stopwords (most c

In [7]:
# Sample records
print("\nSample dim_org records:")
display(dim_org_df[['unique_id', 'name', 'country_code', 'ror_id']].head(5))

print("\nSample GRID records:")
display(grid_df[['unique_id', 'name', 'country_code', 'ror_id']].head(5))



Sample dim_org records:


,unique_id,name,country_code,ror_id
0,dim_ASC-OR-0000000000004-1.0-1724880238,Living Tongues Institute for Endangered Languages,US,https://ror.org/00gens972
1,dim_ASC-OR-0000000000012-1.0-1724880238,International Association of Comparative Korean Studies,KR,https://ror.org/004427313
2,dim_ASC-OR-0000000000015-1.0-1724880238,Sunol Sciences Corporation (United States),US,https://ror.org/002a8dp29
3,dim_ASC-OR-0000000000025-1.0-1724880238,Edelweiss Gestão Empresarial (Brazil),BR,https://ror.org/00avzve25
4,dim_ASC-OR-0000000000027-1.0-1724880238,United Leukodystrophy Foundation,US,https://ror.org/00am1hd38



Sample GRID records:


,unique_id,name,country_code,ror_id
0,grid_grid.425505.3,National Museums of Kenya,KE,https://ror.org/04sjpp691
1,grid_grid.463047.2,African Union Interafrican Bureau for Animal Resources,KE,https://ror.org/01jf9dm55
2,grid_grid.484130.a,Alliance for a Green Revolution in Africa,KE,https://ror.org/01cmcvq49
3,grid_grid.449195.4,Multimedia University of Kenya,KE,https://ror.org/028r4zr88
4,grid_grid.10604.33,University of Nairobi,KE,https://ror.org/02y9nww90


---
## 3. Create Ground Truth Pairs


In [8]:
# Create ground truth from ROR ID matches
positive_pairs, negative_pairs = create_ground_truth_pairs(dim_org_df, grid_df)

print("\nGROUND TRUTH SUMMARY")
print("=" * 50)
print(f"Positive pairs (same ROR): {len(positive_pairs):,}")
print(f"Negative pairs (diff ROR): {len(negative_pairs):,}")


[13:56:32]  Creating ground truth pairs from ROR matches...
[13:56:32]    dim_org with ROR: 105,553
[13:56:32]    GRID with ROR: 102,173
[13:56:32]    Positive pairs: 97,782
[13:56:32]    Negative pairs: 100,000

GROUND TRUTH SUMMARY
Positive pairs (same ROR): 97,782
Negative pairs (diff ROR): 100,000


In [9]:
# Show sample positive pairs (matches)
print("\nSAMPLE POSITIVE PAIRS (True Matches)")
print("=" * 70)

for i, (_, pair) in enumerate(positive_pairs.head(5).iterrows()):
    dim_row = dim_org_df[dim_org_df['unique_id'] == pair['unique_id_l']].iloc[0]
    grid_row = grid_df[grid_df['unique_id'] == pair['unique_id_r']].iloc[0]
    
    print(f"\nPair {i+1}: ROR={pair['ror_id']}")
    print(f"  dim_org: {dim_row['name'][:60]}")
    print(f"  GRID:    {grid_row['name'][:60]}")



SAMPLE POSITIVE PAIRS (True Matches)

Pair 1: ROR=https://ror.org/00gens972
  dim_org: Living Tongues Institute for Endangered Languages
  GRID:    Living Tongues Institute for Endangered Languages

Pair 2: ROR=https://ror.org/004427313
  dim_org: International Association of Comparative Korean Studies
  GRID:    International Association of Comparative Korean Studies

Pair 3: ROR=https://ror.org/002a8dp29
  dim_org: Sunol Sciences Corporation (United States)
  GRID:    Sunol Sciences Corporation (United States)

Pair 4: ROR=https://ror.org/00avzve25
  dim_org: Edelweiss Gestão Empresarial (Brazil)
  GRID:    Edelweiss Gestão Empresarial (Brazil)

Pair 5: ROR=https://ror.org/00am1hd38
  dim_org: United Leukodystrophy Foundation
  GRID:    United Leukodystrophy Foundation


In [10]:
# Show sample negative pairs (non-matches)
print("\nSAMPLE NEGATIVE PAIRS (Non-Matches)")
print("=" * 70)

for i, (_, pair) in enumerate(negative_pairs.head(5).iterrows()):
    dim_row = dim_org_df[dim_org_df['unique_id'] == pair['unique_id_l']].iloc[0]
    grid_row = grid_df[grid_df['unique_id'] == pair['unique_id_r']].iloc[0]
    
    print(f"\nPair {i+1}:")
    print(f"  dim_org: {dim_row['name'][:60]} (ROR: {pair.get('ror_id_l', 'N/A')})")
    print(f"  GRID:    {grid_row['name'][:60]} (ROR: {pair.get('ror_id_r', 'N/A')})")



SAMPLE NEGATIVE PAIRS (Non-Matches)

Pair 1:
  dim_org: Poliklinik für Präventive Zahnheilkunde und Kinderzahnheilku (ROR: https://ror.org/02mj1v005)
  GRID:    Sekolah Tinggi Agama Kristen Teruna Bhakti (ROR: https://ror.org/00wryzy45)

Pair 2:
  dim_org: Spesialprodukter Sør (Norway) (ROR: https://ror.org/0559rdv60)
  GRID:    University of California, San Diego (ROR: https://ror.org/0168r3w48)

Pair 3:
  dim_org: Daniels Fund (ROR: https://ror.org/0536y0s22)
  GRID:    Committee for Economic Development (ROR: https://ror.org/050nvag51)

Pair 4:
  dim_org: University of Kabianga (ROR: https://ror.org/03rk9qf06)
  GRID:    IEEE Foundation (ROR: https://ror.org/040d52e71)

Pair 5:
  dim_org: Tokyo University of the Arts (ROR: https://ror.org/00y809n33)
  GRID:    Doctors Hospital at Renaissance (ROR: https://ror.org/00yh56t79)


---
## 4. Define Splink Settings


In [11]:
# Define comparison functions
# Each comparison defines how to compare a specific field

print("DEFINING COMPARISONS")
print("=" * 50)

# Name comparison with graduated thresholds AND term frequency adjustments
# TF adjustments downweight matches on common terms like "Korean", "University"
# Note: term_frequency_adjustments is set via .configure() method per Splink docs
name_comparison = cl.JaroWinklerAtThresholds(
    "name_normalized",
    [0.95, 0.88, 0.80],
).configure(term_frequency_adjustments=True)  # KEY: Downweight common terms
print("  [x] Name comparison (Jaro-Winkler + TF adjustments)")

# Phonetic comparison on DISTINCTIVE token (not first word)
# This avoids all "Korean Society of X" orgs getting same Soundex code
phonetic_comparison = cl.CustomComparison(
    comparison_levels=[
        cll.NullLevel("distinctive_soundex"),
        cll.ExactMatchLevel("distinctive_soundex"),
        cll.ElseLevel(),
    ],
    output_column_name="phonetic_match",
)
print("  [x] Phonetic comparison (on distinctive token Soundex)")

# Distinctive tokens comparison - checks overlap of rare/important words
# "Korean Society of Systematic Theology" vs "Korean Society of Radiology"
# will have LOW overlap: [systematic, theology] vs [radiology]
# NOTE: Short names like "Pfizer" now get the name itself as distinctive token (data_prep fix)
distinctive_comparison = cl.CustomComparison(
    comparison_levels=[
        cll.NullLevel("distinctive_tokens"),
        # Empty array on either side = neutral (not negative) - safety net for very short names
        {
            "sql_condition": "array_length(distinctive_tokens_l, 1) = 0 OR array_length(distinctive_tokens_r, 1) = 0",
            "label_for_charts": "Empty array (short name)",
        },
        cll.ArrayIntersectLevel("distinctive_tokens", min_intersection=2),  # 2+ shared rare tokens
        cll.ArrayIntersectLevel("distinctive_tokens", min_intersection=1),  # 1 shared rare token  
        cll.ElseLevel(),  # No shared rare tokens - negative evidence
    ],
    output_column_name="distinctive_match",
)
print("  [x] Distinctive tokens comparison (ArrayIntersect + empty array handling)")

# Country comparison
country_comparison = cl.CustomComparison(
    comparison_levels=[
        cll.NullLevel("country_code"),
        cll.ExactMatchLevel("country_code"),
        cll.ElseLevel(),
    ],
    output_column_name="country_match",
)
print("  [x] Country comparison (exact match)")

# City comparison with fuzzy matching
city_comparison = cl.CustomComparison(
    comparison_levels=[
        cll.NullLevel("city"),
        cll.ExactMatchLevel("city"),
        cll.JaroWinklerLevel("city", 0.9),
        cll.ElseLevel(),
    ],
    output_column_name="city_match",
)
print("  [x] City comparison (fuzzy)")

# Asymmetric containment detection
# Catches: "Daegu University" (2 tokens) vs "Daegu University of Foreign Studies" (4 tokens)
# where smaller is proper subset of larger - these are usually DIFFERENT entities
containment_comparison = cl.CustomComparison(
    comparison_levels=[
        cll.NullLevel("name_tokens"),
        
        # Level 1: Exact same tokens = strong positive
        {
            "sql_condition": "list_sort(name_tokens_l) = list_sort(name_tokens_r)",
            "label_for_charts": "Exact token match",
        },
        
        # Level 2: Similar token count AND high overlap = positive
        # (catches legitimate variations like word order)
        {
            "sql_condition": """
                ABS(len(name_tokens_l) - len(name_tokens_r)) <= 1
                AND len(list_intersect(name_tokens_l, name_tokens_r)) >= 
                    GREATEST(len(name_tokens_l), len(name_tokens_r)) - 1
            """,
            "label_for_charts": "Similar tokens (diff <= 1)",
        },
        
        # Level 3: Short name exception - if one name has <= 2 tokens, be lenient
        {
            "sql_condition": """
                (len(name_tokens_l) <= 2 OR len(name_tokens_r) <= 2)
                AND len(list_intersect(name_tokens_l, name_tokens_r)) >= 1
            """,
            "label_for_charts": "Short name with overlap",
        },
        
        # Level 4: One is proper subset of other = SUSPICIOUS (will get low m-prob)
        # This is the key level that catches false positives like "Daegu U" vs "Daegu U of Foreign Studies"
        {
            "sql_condition": """
                (list_sort(list_intersect(name_tokens_l, name_tokens_r)) = list_sort(name_tokens_l)
                 AND len(name_tokens_l) < len(name_tokens_r))
                OR
                (list_sort(list_intersect(name_tokens_l, name_tokens_r)) = list_sort(name_tokens_r)
                 AND len(name_tokens_r) < len(name_tokens_l))
            """,
            "label_for_charts": "Subset containment (suspicious)",
        },
        
        # Level 5: Partial overlap (neither is subset) = neutral
        cll.ElseLevel(),
    ],
    output_column_name="containment_check",
)
print("  [x] Containment detection (subset check)")

# First (most distinctive) token comparison with Jaro-Winkler fuzzy matching
# The MOST distinctive token (index 0) should match for strong positive evidence
# Includes fuzzy matching for slight spelling variations
# "demographic" vs "ecology" = NO match = strong negative evidence
# This catches Max Planck / Fraunhofer institute false positives
first_token_comparison = cl.CustomComparison(
    comparison_levels=[
        cll.NullLevel("distinctive_token"),
        cll.ExactMatchLevel("distinctive_token"),
        cll.JaroWinklerLevel("distinctive_token", 0.9),  # Fuzzy match for typos
        cll.JaroWinklerLevel("distinctive_token", 0.8),  # Looser fuzzy match
        cll.ElseLevel(),  # Different distinctive tokens = strong negative
    ],
    output_column_name="first_token_match",
)
print("  [x] First token comparison (with Jaro-Winkler fuzzy matching)")

# Token overlap penalty - explicit check for ZERO overlap in distinctive tokens
token_overlap_comparison = cl.CustomComparison(
    comparison_levels=[
        cll.NullLevel("distinctive_tokens"),
        # High overlap = strong positive (trainable)
        cll.ArrayIntersectLevel("distinctive_tokens", min_intersection=2),
        # Some overlap = moderate positive (trainable)
        cll.ArrayIntersectLevel("distinctive_tokens", min_intersection=1),
        # Zero overlap = else level (trainable via geographic-only blocking)
        cll.ElseLevel(),
    ],
    output_column_name="token_overlap",
)
print("  [x] Token overlap comparison (fully trainable via EM)")

# Alias matching - name matches any alias in the other record
# Gives strong positive evidence when org name matches a known alias
# Note: all_names contains original names, so compare with original 'name' field
# Use LOWER() for case-insensitive matching
alias_comparison = cl.CustomComparison(
    comparison_levels=[
        cll.NullLevel("all_names"),
        # Name matches an alias exactly (case-insensitive)
        {
            "sql_condition": """
                list_contains(list_transform(all_names_r, x -> LOWER(x)), LOWER(name_l)) OR 
                list_contains(list_transform(all_names_l, x -> LOWER(x)), LOWER(name_r))
            """,
            "label_for_charts": "Name matches alias (exact)",
        },
        cll.ElseLevel(),
    ],
    output_column_name="alias_match",
)
print("  [x] Alias matching (name matches known alias)")

comparisons = [
    name_comparison,
    phonetic_comparison,
    distinctive_comparison,  # Rare token overlap (with empty array handling)
    token_overlap_comparison,  # Explicit zero overlap penalty with FIXED weight
    containment_comparison,  # Asymmetric containment detection
    first_token_comparison,  # Enhanced with Jaro-Winkler
    alias_comparison,  # NEW: Name matches alias
    country_comparison,
    city_comparison,
]


DEFINING COMPARISONS
  [x] Name comparison (Jaro-Winkler + TF adjustments)
  [x] Phonetic comparison (on distinctive token Soundex)
  [x] Distinctive tokens comparison (ArrayIntersect + empty array handling)
  [x] Country comparison (exact match)
  [x] City comparison (fuzzy)
  [x] Containment detection (subset check)
  [x] First token comparison (with Jaro-Winkler fuzzy matching)
  [x] Token overlap comparison (fully trainable via EM)
  [x] Alias matching (name matches known alias)


In [12]:
# Define blocking rules
# Using compound rules to avoid pair explosion from broad single-field matches
# TIGHTENED: Use longer prefix (20 chars) to avoid blocking all "university..." orgs together
print("\nDEFINING BLOCKING RULES")
print("=" * 50)

blocking_rules = [
    # Rule 1: Exact name match
    "l.name_normalized = r.name_normalized",
    # Rule 2: Long prefix + country
    "substr(l.name_normalized, 1, 25) = substr(r.name_normalized, 1, 25) AND l.country_code = r.country_code",
    # Rule 3: Distinctive token + city (NOT just length!)
    "l.distinctive_token = r.distinctive_token AND l.city = r.city",
    # Rule 4: Phonetic + country + city
    "l.name_metaphone = r.name_metaphone AND l.country_code = r.country_code AND l.city = r.city",
]
for i, rule in enumerate(blocking_rules):
    print(f"  [{i+1}] {rule}")



DEFINING BLOCKING RULES
  [1] l.name_normalized = r.name_normalized
  [2] substr(l.name_normalized, 1, 25) = substr(r.name_normalized, 1, 25) AND l.country_code = r.country_code
  [3] l.distinctive_token = r.distinctive_token AND l.city = r.city
  [4] l.name_metaphone = r.name_metaphone AND l.country_code = r.country_code AND l.city = r.city


In [13]:
# Create Splink settings
settings = SettingsCreator(
    link_type="link_only",  # Linking two different datasets
    unique_id_column_name="unique_id",
    comparisons=comparisons,
    blocking_rules_to_generate_predictions=blocking_rules,
)

print("\nSPLINK SETTINGS CREATED")
print("=" * 50)
print(f"Link type: link_only")
print(f"Unique ID column: unique_id")
print(f"Number of comparisons: {len(comparisons)}")
print(f"Number of blocking rules: {len(blocking_rules)}")



SPLINK SETTINGS CREATED
Link type: link_only
Unique ID column: unique_id
Number of comparisons: 9
Number of blocking rules: 4


---
## 5. Initialize Linker


In [14]:
# Initialize Splink Linker with DuckDB backend
with Timer("Initializing Linker"):
    linker = Linker(
        [dim_org_df, grid_df],
        settings,
        db_api=DuckDBAPI()
    )

print("\nLinker initialized successfully")


[13:56:57]  Starting: Initializing Linker
[13:56:57]  Completed: Initializing Linker (0.4s)

Linker initialized successfully


---
## 6. Estimate U-Probabilities (Random Sampling)


In [15]:
# Estimate u-probabilities (probability of random agreement)
# This uses random sampling to estimate how often fields match by chance

with Timer("Estimating u-probabilities"):
    linker.training.estimate_u_using_random_sampling(
        max_pairs=config.sampling.U_PROBABILITY_MAX_PAIRS
    )

print("\nU-probabilities estimated")


----- Estimating u probabilities using random sampling -----


[13:56:59]  Starting: Estimating u-probabilities



Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - name_normalized (no m values are trained).
    - phonetic_match (no m values are trained).
    - distinctive_match (no m values are trained).
    - token_overlap (no m values are trained).
    - containment_check (no m values are trained).
    - first_token_match (no m values are trained).
    - alias_match (no m values are trained).
    - country_match (no m values are trained).
    - city_match (no m values are trained).


[13:57:10]  Completed: Estimating u-probabilities (10.9s)

U-probabilities estimated


In [16]:
# Display u-probability estimates
print("U-PROBABILITY ESTIMATES")
print("=" * 70)
print("(Probability of random/chance agreement for each comparison level)")
print()

# Get model parameters
model_params = linker.misc.save_model_to_json()

for comparison in model_params.get('comparisons', []):
    col_name = comparison.get('output_column_name', 'unknown')
    print(f"\n{col_name}:")
    for level in comparison.get('comparison_levels', []):
        label = level.get('label_for_charts', 'N/A')
        u_prob = level.get('u_probability', 0)
        print(f"  {label:<40} u={u_prob:.6f}")


U-PROBABILITY ESTIMATES
(Probability of random/chance agreement for each comparison level)


name_normalized:
  name_normalized is NULL                  u=0.000000
  Exact match on name_normalized           u=0.000010
  Jaro-Winkler distance of name_normalized >= 0.95 u=0.000002
  Jaro-Winkler distance of name_normalized >= 0.88 u=0.000120
  Jaro-Winkler distance of name_normalized >= 0.8 u=0.001575
  All other comparisons                    u=0.998293

phonetic_match:
  distinctive_soundex is NULL              u=0.000000
  Exact match on distinctive_soundex       u=0.001018
  All other comparisons                    u=0.998982

distinctive_match:
  distinctive_tokens is NULL               u=0.000000
  Empty array (short name)                 u=0.001314
  Array intersection size >= 2             u=0.000035
  Array intersection size >= 1             u=0.001993
  All other comparisons                    u=0.996658

token_overlap:
  distinctive_tokens is NULL               u=0.000000
  Ar

---
## 7. EM Training (Expectation Maximization)


In [17]:
# EM training to estimate m-probabilities
# Block on high-confidence fields to create training pairs

with Timer("EM Training - Stage 1 (name_normalized)"):
    linker.training.estimate_parameters_using_expectation_maximisation(
        block_on("name_normalized"),
        fix_u_probabilities=True
    )

print("\nEM Training Stage 1 complete")


[13:57:25]  Starting: EM Training - Stage 1 (name_normalized)



----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l."name_normalized" = r."name_normalized"

Parameter estimates will be made for the following comparison(s):
    - phonetic_match
    - distinctive_match
    - token_overlap
    - containment_check
    - first_token_match
    - alias_match
    - country_match
    - city_match

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - name_normalized

Level All other comparisons on comparison phonetic_match not observed in dataset, unable to train m value

Level All other comparisons on comparison distinctive_match not observed in dataset, unable to train m value

Level Similar tokens (diff <= 1) on comparison containment_check not observed in dataset, unable to train m value

Level Short name with overlap on comparison containment_check not observed in dataset, unable to train m value

Level Subset containment (suspicious) on 

[13:57:27]  Completed: EM Training - Stage 1 (name_normalized) (2.5s)

EM Training Stage 1 complete


In [18]:
# Stage 2: Train on country + name prefix (country alone is too broad - 500M+ pairs)
with Timer("EM Training - Stage 2 (country + name_prefix_5)"):
    linker.training.estimate_parameters_using_expectation_maximisation(
        "l.country_code = r.country_code AND l.name_prefix_5 = r.name_prefix_5",
        fix_u_probabilities=True
    )

print("\nEM Training Stage 2 complete")



----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l.country_code = r.country_code AND l.name_prefix_5 = r.name_prefix_5

Parameter estimates will be made for the following comparison(s):
    - name_normalized
    - phonetic_match
    - distinctive_match
    - token_overlap
    - containment_check
    - first_token_match
    - alias_match
    - city_match

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - country_match


[13:57:31]  Starting: EM Training - Stage 2 (country + name_prefix_5)



Iteration 1: Largest change in params was -0.605 in the m_probability of containment_check, level `Exact token match`
Iteration 2: Largest change in params was -0.538 in the m_probability of first_token_match, level `Exact match on distinctive_token`
Iteration 3: Largest change in params was 0.351 in the m_probability of distinctive_match, level `All other comparisons`
Iteration 4: Largest change in params was 0.323 in probability_two_random_records_match
Iteration 5: Largest change in params was 0.091 in probability_two_random_records_match
Iteration 6: Largest change in params was 0.0315 in the m_probability of name_normalized, level `All other comparisons`
Iteration 7: Largest change in params was 0.0143 in the m_probability of name_normalized, level `All other comparisons`
Iteration 8: Largest change in params was 0.00712 in the m_probability of name_normalized, level `All other comparisons`
Iteration 9: Largest change in params was 0.00376 in the m_probability of name_normalized,

[13:57:50]  Completed: EM Training - Stage 2 (country + name_prefix_5) (19.5s)

EM Training Stage 2 complete


In [19]:
# Stage 3: Train on distinctive token match (for better token overlap training)
# This helps train the token_overlap and first_token_match comparisons
with Timer("EM Training - Stage 3 (distinctive_token)"):
    linker.training.estimate_parameters_using_expectation_maximisation(
        "l.distinctive_token = r.distinctive_token",
        fix_u_probabilities=True
    )

print("\nEM Training Stage 3 complete")


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l.distinctive_token = r.distinctive_token

Parameter estimates will be made for the following comparison(s):
    - name_normalized
    - phonetic_match
    - distinctive_match
    - token_overlap
    - containment_check
    - alias_match
    - country_match
    - city_match

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - first_token_match


[13:57:53]  Starting: EM Training - Stage 3 (distinctive_token)



Level All other comparisons on comparison phonetic_match not observed in dataset, unable to train m value

Level Empty array (short name) on comparison distinctive_match not observed in dataset, unable to train m value

Level All other comparisons on comparison distinctive_match not observed in dataset, unable to train m value

Level All other comparisons on comparison token_overlap not observed in dataset, unable to train m value

Iteration 1: Largest change in params was -0.912 in the m_probability of phonetic_match, level `All other comparisons`
Iteration 2: Largest change in params was 3.52e-06 in probability_two_random_records_match

EM converged after 2 iterations
m probability not trained for phonetic_match - All other comparisons (comparison vector value: 0). This usually means the comparison level was never observed in the training data.
m probability not trained for distinctive_match - Empty array (short name) (comparison vector value: 3). This usually means the comparison l

[13:57:57]  Completed: EM Training - Stage 3 (distinctive_token) (3.5s)

EM Training Stage 3 complete


In [20]:
# Stage 4: Train on city + name prefix (for geographic diversity)
# This helps train city_match and provides diverse training pairs
with Timer("EM Training - Stage 4 (city + name_prefix_5)"):
    linker.training.estimate_parameters_using_expectation_maximisation(
        "l.city = r.city AND l.name_prefix_5 = r.name_prefix_5",
        fix_u_probabilities=True
    )

print("\nEM Training Stage 4 complete")



----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l.city = r.city AND l.name_prefix_5 = r.name_prefix_5

Parameter estimates will be made for the following comparison(s):
    - name_normalized
    - phonetic_match
    - distinctive_match
    - token_overlap
    - containment_check
    - first_token_match
    - alias_match
    - country_match

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - city_match


[13:58:16]  Starting: EM Training - Stage 4 (city + name_prefix_5)



Iteration 1: Largest change in params was -0.7 in the m_probability of phonetic_match, level `Exact match on distinctive_soundex`
Iteration 2: Largest change in params was 0.133 in probability_two_random_records_match
Iteration 3: Largest change in params was 0.0743 in the m_probability of name_normalized, level `All other comparisons`
Iteration 4: Largest change in params was 0.0734 in probability_two_random_records_match
Iteration 5: Largest change in params was 0.0773 in probability_two_random_records_match
Iteration 6: Largest change in params was 0.0609 in probability_two_random_records_match
Iteration 7: Largest change in params was 0.0337 in probability_two_random_records_match
Iteration 8: Largest change in params was 0.0145 in probability_two_random_records_match
Iteration 9: Largest change in params was 0.00557 in probability_two_random_records_match
Iteration 10: Largest change in params was 0.00218 in probability_two_random_records_match
Iteration 11: Largest change in par

[13:58:24]  Completed: EM Training - Stage 4 (city + name_prefix_5) (8.7s)

EM Training Stage 4 complete


In [21]:
# Stage 5: Geographic-only blocking (trains zero-overlap scenarios)
# This creates pairs where names DON'T overlap but location matches
with Timer("EM Training - Stage 5 (country + city only)"):
    linker.training.estimate_parameters_using_expectation_maximisation(
        "l.country_code = r.country_code AND l.city = r.city",
        fix_u_probabilities=True
    )


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l.country_code = r.country_code AND l.city = r.city

Parameter estimates will be made for the following comparison(s):
    - name_normalized
    - phonetic_match
    - distinctive_match
    - token_overlap
    - containment_check
    - first_token_match
    - alias_match

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - country_match
    - city_match


[13:58:33]  Starting: EM Training - Stage 5 (country + city only)



Iteration 1: Largest change in params was -0.394 in the m_probability of phonetic_match, level `Exact match on distinctive_soundex`
Iteration 2: Largest change in params was -0.137 in the m_probability of distinctive_match, level `All other comparisons`
Iteration 3: Largest change in params was -0.137 in the m_probability of distinctive_match, level `All other comparisons`
Iteration 4: Largest change in params was -0.133 in the m_probability of distinctive_match, level `All other comparisons`
Iteration 5: Largest change in params was -0.104 in the m_probability of distinctive_match, level `All other comparisons`
Iteration 6: Largest change in params was -0.0611 in the m_probability of distinctive_match, level `All other comparisons`
Iteration 7: Largest change in params was -0.0245 in the m_probability of token_overlap, level `All other comparisons`
Iteration 8: Largest change in params was -0.00743 in the m_probability of token_overlap, level `All other comparisons`
Iteration 9: Larg

[14:00:45]  Completed: EM Training - Stage 5 (country + city only) (131.8s)


In [ ]:
# Display m-probability estimates (match probabilities)
print("M-PROBABILITY ESTIMATES")
print("=" * 70)
print("(Probability of agreement given records ARE a match)")
print()

model_params = linker.misc.save_model_to_json()

for comparison in model_params.get('comparisons', []):
    col_name = comparison.get('output_column_name', 'unknown')
    print(f"\n{col_name}:")
    for level in comparison.get('comparison_levels', []):
        label = level.get('label_for_charts', 'N/A')
        m_prob = level.get('m_probability', 0)
        u_prob = level.get('u_probability', 0)
        
        # Calculate match weight
        if u_prob > 0 and m_prob > 0:
            weight = np.log2(m_prob / u_prob)
        else:
            weight = 0
            
        print(f"  {label:<40} m={m_prob:.4f} u={u_prob:.6f} weight={weight:+.2f}")


In [ ]:
# Estimate and set prior probability based on ground truth
# This calibrates match scores to be more meaningful
print("PRIOR PROBABILITY ESTIMATION")
print("=" * 70)

n_positive = len(positive_pairs)
n_dim_org = len(dim_org_df)
n_grid = len(grid_df)
n_total_possible = n_dim_org * n_grid
estimated_prior = n_positive / n_total_possible

print(f"Known positive pairs:     {n_positive:,}")
print(f"Total possible pairs:     {n_total_possible:,}")
print(f"Estimated prior:          {estimated_prior:.8f}")
print(f"Default Splink prior:     0.0001")

# Use a MORE CONSERVATIVE prior to push uncertain pairs toward lower scores
# This helps create a more bimodal distribution (fewer pairs in 0.5-0.85 gray zone)
conservative_prior = 1e-6  # More conservative than estimated
print(f"Using conservative prior: {conservative_prior:.8f}")

# Update linker with conservative prior
linker._settings_obj._probability_two_random_records_match = conservative_prior
print(f"\nUpdated linker with conservative prior probability")


In [ ]:
linker.visualisations.match_weights_chart()

---
## 8. Validation on Predictions


In [ ]:
# Debug: Visualize cumulative blocking rule comparisons
# This shows how many pairs each rule generates (cumulative)
print("BLOCKING RULE PAIR ANALYSIS")
print("=" * 70)

try:
    # Splink 4.x: Use the visualization method
    chart = linker.visualisations.cumulative_num_comparisons_from_blocking_rules_chart()
    display(chart)
except Exception as e:
    print(f"Could not generate blocking chart: {e}")
    print("\nProceeding with predictions - blocking rules are:")
    for i, rule in enumerate(blocking_rules):
        print(f"  [{i+1}] {rule}")


In [ ]:
# Generate predictions
with Timer("Generating predictions"):
    predictions = linker.inference.predict(
        threshold_match_probability=config.matching.THRESHOLD_PREDICTION
    )
    predictions_df = predictions.as_pandas_dataframe()

print(f"\nTotal predictions: {len(predictions_df):,}")


In [ ]:
# Prediction score distribution
print("PREDICTION SCORE DISTRIBUTION")
print("=" * 50)

score_col = 'match_probability'
if score_col in predictions_df.columns:
    bins = [0, 0.5, 0.7, 0.85, 0.95, 1.0]
    labels = ['<0.5', '0.5-0.7', '0.7-0.85', '0.85-0.95', '0.95-1.0']
    predictions_df['score_bin'] = pd.cut(predictions_df[score_col], bins=bins, labels=labels)
    
    print("\nMatch probability distribution:")
    print(predictions_df['score_bin'].value_counts().sort_index())


In [ ]:
# Add ground truth labels to predictions FIRST (before display)
print("ADDING GROUND TRUTH LABELS TO PREDICTIONS")
print("=" * 50)

# Create ground truth lookup
positive_set = set(
    zip(positive_pairs['unique_id_l'], positive_pairs['unique_id_r'])
)
negative_set = set(
    zip(negative_pairs['unique_id_l'], negative_pairs['unique_id_r'])
)

def get_truth_label(row):
    pair = (row['unique_id_l'], row['unique_id_r'])
    pair_rev = (row['unique_id_r'], row['unique_id_l'])
    
    if pair in positive_set or pair_rev in positive_set:
        return 'TRUE_MATCH'
    elif pair in negative_set or pair_rev in negative_set:
        return 'TRUE_NON_MATCH'
    else:
        return 'UNKNOWN'

predictions_df['truth_label'] = predictions_df.apply(get_truth_label, axis=1)

# Summary
print(predictions_df['truth_label'].value_counts())

# Sample high-confidence predictions - LABELED PAIRS FIRST
print("\nSAMPLE HIGH-CONFIDENCE PREDICTIONS (>0.95)")
print("=" * 70)

# Show labeled pairs first, then UNKNOWN
high_conf_labeled = predictions_df[
    (predictions_df['match_probability'] > 0.95) & 
    (predictions_df['truth_label'] != 'UNKNOWN')
].head(15)

high_conf_unknown = predictions_df[
    (predictions_df['match_probability'] > 0.95) & 
    (predictions_df['truth_label'] == 'UNKNOWN')
].head(5)

high_conf = pd.concat([high_conf_labeled, high_conf_unknown])

for _, pred in high_conf.iterrows():
    left_id = pred['unique_id_l']
    right_id = pred['unique_id_r']
    prob = pred['match_probability']
    label = pred['truth_label']
    
    # Get names
    left_name = dim_org_df[dim_org_df['unique_id'] == left_id]['name'].iloc[0] if left_id in dim_org_df['unique_id'].values else 'N/A'
    right_name = grid_df[grid_df['unique_id'] == right_id]['name'].iloc[0] if right_id in grid_df['unique_id'].values else 'N/A'
    
    # Color code: TRUE_MATCH = good, TRUE_NON_MATCH = FALSE POSITIVE!
    status = "FP!" if label == 'TRUE_NON_MATCH' else ("OK" if label == 'TRUE_MATCH' else "?")
    
    print(f"\nProb: {prob:.4f} [{status}] {label}")
    print(f"  L: {left_name[:60]}")
    print(f"  R: {right_name[:60]}")

In [ ]:
high_conf

In [ ]:
# Show false positives (predicted match but truth is non-match)
# Note: Ground truth labels were added in the cell above
high_conf_fp = predictions_df[
    (predictions_df['match_probability'] >= 0.95) & 
    (predictions_df['truth_label'] == 'TRUE_NON_MATCH')
]
print(f"FALSE POSITIVES (prob >= 0.95, truth = non-match): {len(high_conf_fp)}")

if len(high_conf_fp) > 0:
    print("\nFalse Positive Details:")
    for idx, row in high_conf_fp.head(10).iterrows():
        print(f"\n  Prob: {row['match_probability']:.4f}")
        print(f"  L: {row['name_normalized_l'][:50]}")
        print(f"  R: {row['name_normalized_r'][:50]}")
        print(f"  gamma_first_token_match: {row['gamma_first_token_match']}")

In [ ]:
high_conf_fp

In [ ]:
# DEBUG: Inspect feature values for specific pairs
print("FEATURE INSPECTION FOR SUSPICIOUS PAIRS")
print("=" * 70)

# Get predictions with comparison details
df = predictions_df.copy()

# Filter to Max Planck cases
mp_pairs = df[
    (df['name_normalized_l'].str.contains('max planck', case=False, na=False)) &
    (df['name_normalized_r'].str.contains('max planck', case=False, na=False)) &
    (df['name_normalized_l'] != df['name_normalized_r'])
].head(5)

# Show all columns for these pairs
for idx, row in mp_pairs.iterrows():
    print(f"\nPair: {row['match_probability']:.4f}")
    print(f"  L: {row['name_normalized_l'][:60]}")
    print(f"  R: {row['name_normalized_r'][:60]}")
    
    # Show comparison outputs (gamma values)
    gamma_cols = [c for c in row.index if c.startswith('gamma_')]
    for col in gamma_cols:
        print(f"  {col}: {row[col]}")
    
    # Show distinctive tokens if available
    if 'distinctive_tokens_l' in row.index:
        print(f"  distinctive_tokens_l: {row['distinctive_tokens_l']}")
        print(f"  distinctive_tokens_r: {row['distinctive_tokens_r']}")

In [ ]:
# Sample low-confidence predictions (near threshold)
print("\nSAMPLE LOW-CONFIDENCE PREDICTIONS (0.5-0.7)")
print("=" * 70)

low_conf = predictions_df[
    (predictions_df['match_probability'] >= 0.50) & 
    (predictions_df['match_probability'] < 0.70)
].head(10)

for _, pred in low_conf.iterrows():
    left_id = pred['unique_id_l']
    right_id = pred['unique_id_r']
    prob = pred['match_probability']
    
    # Get names
    left_name = dim_org_df[dim_org_df['unique_id'] == left_id]['name'].iloc[0] if left_id in dim_org_df['unique_id'].values else 'N/A'
    right_name = grid_df[grid_df['unique_id'] == right_id]['name'].iloc[0] if right_id in grid_df['unique_id'].values else 'N/A'
    
    print(f"\nProb: {prob:.4f}")
    print(f"  L: {left_name[:60]}")
    print(f"  R: {right_name[:60]}")

In [ ]:
# ACCURACY EVALUATION USING GROUND TRUTH LABELS
print("MODEL ACCURACY EVALUATION")
print("=" * 70)

# Combine ground truth
ground_truth = pd.concat([
    positive_pairs[['unique_id_l', 'unique_id_r', 'is_match']],
    negative_pairs[['unique_id_l', 'unique_id_r', 'is_match']]
])
print(f"Total ground truth pairs: {len(ground_truth):,}")
print(f"  Positive (true matches): {ground_truth['is_match'].sum():,}")
print(f"  Negative (non-matches):  {(~ground_truth['is_match']).sum():,}")

# Join with predictions
ground_truth['pair_key'] = ground_truth['unique_id_l'] + '|' + ground_truth['unique_id_r']
predictions_df['pair_key'] = predictions_df['unique_id_l'] + '|' + predictions_df['unique_id_r']

# Merge
eval_df = ground_truth.merge(
    predictions_df[['pair_key', 'match_probability']],
    on='pair_key',
    how='left'
)

# Pairs not in predictions = not blocked = predicted as non-match
eval_df['match_probability'] = eval_df['match_probability'].fillna(0)

print(f"\nPairs with predictions: {(eval_df['match_probability'] > 0).sum():,}")
print(f"Pairs not blocked (score=0): {(eval_df['match_probability'] == 0).sum():,}")

# Calculate metrics at different thresholds
print("\n" + "=" * 70)
print("METRICS AT DIFFERENT THRESHOLDS")
print("=" * 70)

thresholds = [0.5, 0.7, 0.85, 0.95]
for thresh in thresholds:
    pred_positive = eval_df['match_probability'] >= thresh
    actual_positive = eval_df['is_match']
    
    tp = ((actual_positive) & (pred_positive)).sum()
    fp = ((~actual_positive) & (pred_positive)).sum()
    tn = ((~actual_positive) & (~pred_positive)).sum()
    fn = ((actual_positive) & (~pred_positive)).sum()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"\nThreshold >= {thresh}:")
    print(f"  TP: {tp:,}  FP: {fp:,}  TN: {tn:,}  FN: {fn:,}")
    print(f"  Precision: {precision:.4f}  (of predicted matches, how many are correct)")
    print(f"  Recall:    {recall:.4f}  (of true matches, how many did we find)")
    print(f"  F1 Score:  {f1:.4f}")

---
## 9. Save Model


In [ ]:
# Save trained model
import json
from pathlib import Path

model_path = config.paths.MODEL_FILE
Path(model_path).parent.mkdir(parents=True, exist_ok=True)

# Get model as JSON
model_json = linker.misc.save_model_to_json()

# Add metadata
model_json['training_metadata'] = {
    'timestamp': pd.Timestamp.now().isoformat(),
    'dim_org_records': len(dim_org_df),
    'grid_records': len(grid_df),
    'positive_pairs': len(positive_pairs),
    'negative_pairs': len(negative_pairs),
    'total_predictions': len(predictions_df)
}

# Save
with open(model_path, 'w') as f:
    json.dump(model_json, f, indent=2)

print(f"Model saved to: {model_path}")


---
## 10. Training Summary


In [ ]:
# Training summary
print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print(f"""
TRAINING DATA:
  - dim_organization: {len(dim_org_df):,} records
  - GRID: {len(grid_df):,} records
  
GROUND TRUTH:
  - Positive pairs: {len(positive_pairs):,}
  - Negative pairs: {len(negative_pairs):,}

PREDICTIONS:
  - Total: {len(predictions_df):,}
  - High confidence (>0.95): {(predictions_df['match_probability'] > 0.95).sum():,}
  - Medium confidence (0.7-0.95): {((predictions_df['match_probability'] > 0.7) & (predictions_df['match_probability'] <= 0.95)).sum():,}

MODEL SAVED:
  - Path: {config.paths.MODEL_FILE}

NEXT STEPS:
  1. Review predictions in 4_analysis.ipynb
  2. Run inference on full dataset in 3_inference.ipynb
""")

# Cleanup
db.close()
print("Database connection closed")
